# Laboratory Task 6: Converting CNN Architecture Diagram into PyTorch

<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Activity 6</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Convert the following CNN architecture diagram into a PyTorch CNN Architecture.
</p>

<center><img src="figures/quick_draw.png" width="400px"></center>
</div>

### Architectural Analysis and Dimensional Calculations

Using the convolution and pooling output formula:
$$\text{Output Shape} = \left\lfloor \frac{W - F + 2P}{S} \right\rfloor + 1$$

| Stage / Layer | Specifications | Dimension Formula | Output Tensor Shape $(B, C, H, W)$ |
| :--- | :--- | :--- | :--- |
| **Input** | Grayscale sketch images | Given input | `(32, 1, 28, 28)` |
| **Conv1 + ReLU** | $K=3\times3, S=1, P=1, C_{out}=32$ | $(28 - 3 + 2(1))/1 + 1 = 28$ | `(32, 32, 28, 28)` |
| **MaxPool1** | $K=2\times2, S=2, P=1$ | $(28 - 2 + 2(1))//2 + 1 = 15$ | `(32, 32, 15, 15)` |
| **Conv2 + ReLU** | $K=3\times3, S=1, P=1, C_{out}=64$ | $(15 - 3 + 2(1))/1 + 1 = 15$ | `(32, 64, 15, 15)` |
| **Conv3 + ReLU** | $K=3\times3, S=1, P=1, C_{out}=128$ | $(15 - 3 + 2(1))/1 + 1 = 15$ | `(32, 128, 15, 15)` |
| **Conv4 + ReLU** | $K=3\times3, S=1, P=1, C_{out}=256$ | $(15 - 3 + 2(1))/1 + 1 = 15$ | `(32, 256, 15, 15)` |
| **MaxPool2** | $K=2\times2, S=2, P=0$ | $(15 - 2 + 0)//2 + 1 = 7$ | `(32, 256, 7, 7)` |
| **Dropout** | Dropout probability $p = 0.2$ | Preserves feature shape | `(32, 256, 7, 7)` |
| **Flatten** | Reshape spatial grid into 1D vector | $256 \times 7 \times 7 = 12,544$ | `(32, 12544)` |
| **FCN1 + ReLU** | Linear: $in=12544, out=1000$ | Fully connected projection | `(32, 1000)` |
| **FCN2 + ReLU** | Linear: $in=1000, out=500$ | Fully connected projection | `(32, 500)` |
| **FCN3 + Softmax** | Linear: $in=500, out=10$ | Class probability distribution | `(32, 10)` |

### Parameter Calculation Breakdown

$$\begin{align*}
\text{Conv1 Parameters} &= (1 \times 32 \times 3 \times 3) + 32 = 288 + 32 = 320 \\
\text{Conv2 Parameters} &= (32 \times 64 \times 3 \times 3) + 64 = 18,432 + 64 = 18,496 \\
\text{Conv3 Parameters} &= (64 \times 128 \times 3 \times 3) + 128 = 73,728 + 128 = 73,856 \\
\text{Conv4 Parameters} &= (128 \times 256 \times 3 \times 3) + 256 = 294,912 + 256 = 295,168 \\
\text{FCN1 Parameters} &= (12,544 \times 1,000) + 1,000 = 12,544,000 + 1,000 = 12,545,000 \\
\text{FCN2 Parameters} &= (1,000 \times 500) + 500 = 500,000 + 500 = 500,500 \\
\text{FCN3 Parameters} &= (500 \times 10) + 10 = 5,000 + 10 = 5,010 \\
\mathbf{\text{Total Learnable Parameters}} &= \mathbf{13,438,350}
\end{align*}$$


In [1]:
# 1. Standard Imports
import torch
import torch.nn as nn
import torch.nn.functional as F

# 2. Dimension verification helper from lecture notebook
def calc_out(w, f, s, p):
    """
    Calculate output shape of a matrix after a convolution or pooling layer.
    Formula: ((w - f + 2 * p) // s) + 1
    """
    out_dim = (w - f + 2 * p) // s + 1
    return out_dim

# Verify spatial dimensions along the pipeline
conv1_w = calc_out(28, 3, 1, 1)  # Conv1: 28 -> 28
pool1_w = calc_out(conv1_w, 2, 2, 1)  # MaxPool1 (padding=1): 28 -> 15
conv2_w = calc_out(pool1_w, 3, 1, 1)  # Conv2: 15 -> 15
conv3_w = calc_out(conv2_w, 3, 1, 1)  # Conv3: 15 -> 15
conv4_w = calc_out(conv3_w, 3, 1, 1)  # Conv4: 15 -> 15
pool2_w = calc_out(conv4_w, 2, 2, 0)  # MaxPool2 (padding=0): 15 -> 7

print(f"Spatial dimensions: Input=28 -> Conv1={conv1_w} -> MaxPool1={pool1_w} -> Conv4={conv4_w} -> MaxPool2={pool2_w}")
print(f"Flattened feature count: 256 * {pool2_w} * {pool2_w} = {256 * pool2_w * pool2_w}")

# 3. PyTorch CNN Architecture Definition
class QuickDrawCNN(nn.Module):
    """
    PyTorch CNN architecture corresponding to Laboratory Activity 6.
    Translates the complete Quick, Draw! convolutional network diagram.
    """
    def __init__(self, num_classes=10):
        super(QuickDrawCNN, self).__init__()
        
        # Block 1: Conv1 + MaxPool1
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=(3, 3), stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), stride=2, padding=1)
        
        # Block 2: Conv2 + Conv3 + Conv4 + MaxPool2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3, 3), stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(3, 3), stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=2, padding=0)
        
        # Regularization & Flatten
        self.dropout = nn.Dropout(p=0.2)
        
        # Fully Connected Network (FCN)
        # Input to FCN1: 256 channels * 7 * 7 spatial grid = 12,544 units
        self.fcn1 = nn.Linear(in_features=256 * 7 * 7, out_features=1000)
        self.fcn2 = nn.Linear(in_features=1000, out_features=500)
        self.fcn3 = nn.Linear(in_features=500, out_features=num_classes)

    def forward(self, x):
        # Block 1
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        
        # Block 2
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)
        
        # Dropout and Flatten
        x = self.dropout(x)
        x = x.view(-1, 256 * 7 * 7)
        
        # Fully Connected Layers
        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = F.softmax(self.fcn3(x), dim=1)
        
        return x

# 4. Model Instantiation
model = QuickDrawCNN(num_classes=10)
print(model)

# 5. Layer-by-Layer Shape Verification with Batch Size = 32
print("\n=== FORWARD PASS LAYER DIMENSION TRACKING ===")
dummy_input = torch.randn(32, 1, 28, 28)
print("Input Tensor:       ", dummy_input.shape)

# Trace shapes step-by-step
out_c1 = F.relu(model.conv1(dummy_input))
print("After Conv1 + ReLU: ", out_c1.shape)     # expected: (32, 32, 28, 28)
out_p1 = model.pool1(out_c1)
print("After MaxPool1:     ", out_p1.shape)     # expected: (32, 32, 15, 15)

out_c2 = F.relu(model.conv2(out_p1))
print("After Conv2 + ReLU: ", out_c2.shape)     # expected: (32, 64, 15, 15)
out_c3 = F.relu(model.conv3(out_c2))
print("After Conv3 + ReLU: ", out_c3.shape)     # expected: (32, 128, 15, 15)
out_c4 = F.relu(model.conv4(out_c3))
print("After Conv4 + ReLU: ", out_c4.shape)     # expected: (32, 256, 15, 15)
out_p2 = model.pool2(out_c4)
print("After MaxPool2:     ", out_p2.shape)     # expected: (32, 256, 7, 7)

out_drop = model.dropout(out_p2)
out_flat = out_drop.view(-1, 256 * 7 * 7)
print("After Flatten (32,?):", out_flat.shape)   # expected: (32, 12544)

out_fc1 = F.relu(model.fcn1(out_flat))
print("After FCN1 + ReLU:  ", out_fc1.shape)    # expected: (32, 1000)
out_fc2 = F.relu(model.fcn2(out_fc1))
print("After FCN2 + ReLU:  ", out_fc2.shape)    # expected: (32, 500)
out_fc3 = F.softmax(model.fcn3(out_fc2), dim=1)
print("After FCN3 + SoftMax:", out_fc3.shape)  # expected: (32, 10)

# Full Model Pass
output = model(dummy_input)
print("\nFull Model Output Tensor Shape:", output.shape)

# 6. Parameter Counting
def count_parameters(m):
    print("\n=== LEARNABLE PARAMETERS PER LAYER ===")
    total = 0
    for name, param in m.named_parameters():
        if param.requires_grad:
            num = param.numel()
            total += num
            print(f"{name:20s}: {num:>10,d}")
    print("-" * 33)
    print(f"{'Total Parameters':20s}: {total:>10,d}")
    return total

total_params = count_parameters(model)

# 7. Loss Criterion & Optimizer Setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(f"\nLoss Criterion: {criterion}")
print(f"Optimizer:      {optimizer.__class__.__name__} (lr=0.001)")


Spatial dimensions: Input=28 -> Conv1=28 -> MaxPool1=15 -> Conv4=15 -> MaxPool2=7
Flattened feature count: 256 * 7 * 7 = 12544
QuickDrawCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=(2, 2), stride=2, padding=1, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=(2, 2), stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.2, inplace=False)
  (fcn1): Linear(in_features=12544, out_features=1000, bias=True)
  (fcn2): Linear(in_features=1000, out_features=500, bias=True)
  (fcn3): Linear(in_features=500, out_features=10, bias=True)
)

=== FORWARD PASS LAYER DIMENSION TRACKING ===
Input Tensor:        torch.Size([32, 1, 28, 28])
After Conv1 + ReLU:  torch.Size([3